# StormEngine V8 — staged model development

Stage 1 pretrains the mask-aware spatial Encoder and Decoder by reconstructing the current ERA5 grid from the final sparse input hour. It uses the frozen V7-B 390-point contract, 2010–2015 training, and 2016 validation. It does not read 2017 or the August 2026 operational evaluation week.

In [1]:
from pathlib import Path
import json, subprocess, sys, yaml

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO

def run_live(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
    return code

print('Repository:', REPO)

Repository: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL


## 1. Create the ignored Windows configuration
Change only `WINDOWS_ERA5_ROOT` if the data folder moved. The generated `.local.yaml` file is ignored by Git.

In [2]:
WINDOWS_ERA5_ROOT = Path(r'D:\Documents\py_projects\StormEngine-DL\DownloadDate')
assert (WINDOWS_ERA5_ROOT / 'cache' / 'stormengine_2010_2017' / 'metadata.json').exists(), WINDOWS_ERA5_ROOT
local_config = REPO / 'configs' / 'v8_reconstruction_windows.local.yaml'
local_config.write_text(yaml.safe_dump({
    'extends': 'v8_reconstruction.yaml',
    'data': {'era5_root': str(WINDOWS_ERA5_ROOT)},
    'training': {'batch_size': 16, 'num_workers': 0},
}, sort_keys=False), encoding='utf-8')
print('ERA5:', WINDOWS_ERA5_ROOT)
print('Local config:', local_config)

ERA5: D:\Documents\py_projects\StormEngine-DL\DownloadDate
Local config: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_windows.local.yaml


## 2. Verify CUDA

In [3]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.5.1+cu121
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 3. Preflight
This verifies the real 390-point cache contract and one complete forward pass.

In [4]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'preflight', '--config', str(local_config), '--device', DEVICE])

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\train_v8_reconstruction.py preflight --config D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_windows.local.yaml --device cuda


{


  "mode": "preflight",


  "device": "cuda",


  "train_samples": 52567,


...

      "point_static_channels": 2


    }


  }


}


0

## 4. Two-batch smoke run
This checks backward propagation and checkpoint writing. Its metrics are not scientific results.

In [5]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'smoke', '--config', str(local_config), '--device', DEVICE, '--output-dir', 'artifacts/v8_spatial_smoke'])

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\train_v8_reconstruction.py smoke --config D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_windows.local.yaml --device cuda --output-dir artifacts/v8_spatial_smoke


  epoch 1 train: 2/2 loss=0.891408 elapsed=0.0m ETA=0.0m


  epoch 1 validation: 1/1 loss=0.885658 elapsed=0.0m ETA=0.0m


Epoch 001/1: train=0.891408 validation=0.885658 *


{


...

    }


  }


}


Artifacts: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\artifacts\v8_spatial_smoke


0

## 5. Five-epoch capped pilot
Inspect learning direction, speed, and GPU memory before enabling the full run.

In [6]:
run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'pilot', '--config', str(local_config), '--device', DEVICE, '--output-dir', 'artifacts/v8_spatial_pilot'])

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\train_v8_reconstruction.py pilot --config D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_windows.local.yaml --device cuda --output-dir artifacts/v8_spatial_pilot


  epoch 1 train: 100/300 loss=0.987902 elapsed=0.1m ETA=0.3m


  epoch 1 train: 200/300 loss=0.824619 elapsed=0.2m ETA=0.1m


  epoch 1 train: 300/300 loss=0.747444 elapsed=0.3m ETA=0.0m


  epoch 1 validation: 75/75 loss=0.793227 elapsed=0.0m ETA=0.0m


...

    }


  }


}


Artifacts: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\artifacts\v8_spatial_pilot


0

## 6. Full Stage-1 training
Leave this locked until the pilot loss is finite and decreasing. Model selection uses only 2016.

In [7]:
RUN_FULL = False
if RUN_FULL:
    run_live([sys.executable, '-u', str(REPO / 'scripts' / 'train_v8_reconstruction.py'), 'train', '--config', str(local_config), '--device', DEVICE])
else:
    print('Full run locked. Set RUN_FULL=True only after inspecting the pilot.')

Full run locked. Set RUN_FULL=True only after inspecting the pilot.


## 7. Inspect the selected summary
Use the pilot directory first; switch to `v8_spatial_pretraining` after the full run. MSL is reported separately because no same-semantics pressure input exists.

In [8]:
RESULT_NAME = 'v8_spatial_pilot'
summary_name = 'pilot_summary.json' if RESULT_NAME.endswith('pilot') else 'train_summary.json'
summary_path = REPO / 'artifacts' / RESULT_NAME / summary_name
print(json.dumps(json.loads(summary_path.read_text(encoding='utf-8')), indent=2))

{
  "schema_version": 1,
  "mode": "pilot",
  "scientific_status": "pilot_only",
  "processor_used": false,
  "train_years": [
    2010,
    2011,
    2012,
    2013,
    2014,
    2015
  ],
  "validation_years": [
    2016
  ],
  "test_years_read": [],
  "best_epoch": 5,
  "best_validation_loss": 0.6487083425124486,
  "validation_metrics": {
    "full": {
      "msl": {
        "mae": 6.876848016156557,
        "rmse": 8.278150920279028
      },
      "u10": {
        "mae": 1.5745053548477597,
        "rmse": 2.253291240390174
      },
      "v10": {
        "mae": 1.5177104001427686,
        "rmse": 2.039550200665562
      },
      "t2m": {
        "mae": 2.4287203028494155,
        "rmse": 3.178568308665671
      },
      "tp": {
        "mae": 0.15546638612159303,
        "rmse": 0.32728147872638924
      }
    },
    "land": {
      "msl": {
        "mae": 6.9583115458065885,
        "rmse": 8.409283399190896
      },
      "u10": {
        "mae": 1.208467901935263,
        "rmse